<a href="https://colab.research.google.com/github/trma2416/CSCI_5502_Fantasy_Project/blob/main/DataForFantasy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# CSCI 5502: Data Mining
## Semester Group Project: Milestone 2
### Fantasy Football: Data Acquisition, Preparation & Exploration

**University:** University of Colorado Boulder

### Project Overview

This notebook documents the data acquisition, cleaning, preprocessing, and exploratory data analysis (EDA) for our fantasy football project.

Our goal is to create a reliable and reproducible dataset that supports our research questions from Milestone 1.

### Notebook Structure

1. Data Acquisition and Provenance
2. Dataset Overview and Data Dictionary
3. Data Cleaning and Preprocessing
4. Data Quality, Bias, and Limitations
5. Exploratory Data Analysis (EDA)
6. Research Question Updates


In [ ]:
player_name = "James Cook"
player_team = "buf" #team name abbr lower
seasons = [2024, 2025, 2026]

In [ ]:
import time
import requests
import pandas as pd

In [ ]:
HEADERS = {"User-Agent": "Mozilla/5.0"}
SEARCH_URL = "https://site.web.api.espn.com/apis/common/v3/search"
GAMELOG_URL = (
    "https://site.web.api.espn.com/apis/common/v3/sports/football/nfl/athletes/{pid}/gamelog"
)
STANDINGS_URL = "https://site.api.espn.com/apis/v2/sports/football/nfl/standings"
SCHEDULE_URL = "https://site.api.espn.com/apis/site/v2/sports/football/nfl/teams/{team_abbr_lower}/schedule"

In [ ]:
def find_player_id(player_name):
    """Look up an NFL player's ESPN id by name using ESPN's search endpoint."""
    r = requests.get(
        SEARCH_URL,
        headers=HEADERS,
        params={"query": player_name, "type": "player", "limit": 5},
    )
    r.raise_for_status()
    items = [p for p in r.json().get("items", []) if p.get("league") == "nfl"]
    if not items:
        return None
    return items[0]["id"]  # top result, ranked by relevance

In [ ]:
def get_gamelog(player_id, season):
    """Fetch one season of game logs (regular season + postseason) as a DataFrame."""
    r = requests.get(
        GAMELOG_URL.format(pid=player_id),
        headers=HEADERS,
        params={"season": season},
    )
    r.raise_for_status()
    data = r.json()

    names = data["names"]            # unique stat keys (labels repeat YDS/AVG/TD/LNG)
    events_info = data["events"]     # game details keyed by eventId
    rows = []

    for st in data["seasonTypes"]:
        for cat in st.get("categories", []):
            for ev in cat.get("events", []):
                info = events_info.get(ev["eventId"], {})
                row = dict(zip(names, ev["stats"]))
                row["season"] = season
                row["season_type"] = st.get("displayName")
                row["eventId"] = ev["eventId"]
                row["date"] = info.get("gameDate")
                row["opponent"] = info.get("opponent", {}).get("abbreviation")
                row["atVs"] = info.get("atVs")
                row["result"] = info.get("gameResult")
                row["score"] = info.get("score")
                rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
def get_defense_ranks(season):
    """Rank teams by points allowed for a season (1 = fewest allowed)."""
    r = requests.get(STANDINGS_URL, headers=HEADERS, params={"season": season})
    r.raise_for_status()
    pa = {}
    for conf in r.json()["children"]:
        for e in conf["standings"]["entries"]:
            for s in e["stats"]:
                if s.get("name") == "pointsAgainst":
                    pa[e["team"]["abbreviation"]] = s.get("value")
    return pd.Series(pa).rank(method="min").astype(int)


In [ ]:


player_id = find_player_id(player_name)
if player_id is None:
    raise SystemExit(f"Player '{player_name}' not found.")
print(f"Found ID: {player_id}")

frames = []
for year in seasons:
    frames.append(get_gamelog(player_id, year))
    time.sleep(1)  # be polite so ESPN doesn't start returning empty 202s

df = pd.concat(frames, ignore_index=True)

# Keep regular season only
df_reg = df[df["season_type"].str.contains("Regular", na=False)].copy()

ranks = {y: get_defense_ranks(y) for y in df_reg["season"].unique()}
df_reg["opp_def_rank"] = [ranks[s].get(o) for s, o in zip(df_reg["season"], df_reg["opponent"])]

df_reg.head()


Found ID: 4379399


,rushingAttempts,rushingYards,yardsPerRushAttempt,rushingTouchdowns,longRushing,receptions,receivingTargets,receivingYards,yardsPerReception,receivingTouchdowns,...,kicksBlocked,season,season_type,eventId,date,opponent,atVs,result,score,opp_def_rank
3,10,28,2.8,1,7,0,0,0,0.0,0,...,-,2024,2024 Regular Season,401671831,2025-01-05T18:00:00.000+00:00,NE,@,L,23-16,22
4,15,53,3.5,1,8,0,0,0,0.0,0,...,-,2024,2024 Regular Season,401671495,2024-12-29T18:00:00.000+00:00,NYJ,vs,W,40-14,20
5,11,100,9.1,1,46,3,3,26,8.7,1,...,-,2024,2024 Regular Season,401671494,2024-12-22T21:25:00.000+00:00,NE,vs,W,24-21,22
6,14,105,7.5,2,41,1,1,28,28.0,0,...,-,2024,2024 Regular Season,401671681,2024-12-15T21:25:00.000+00:00,DET,@,W,48-42,7
7,6,20,3.3,0,8,2,2,9,4.5,0,...,-,2024,2024 Regular Season,401671773,2024-12-08T21:25:00.000+00:00,LAR,@,L,44-42,17


In [ ]:
def get_next_opponent(team_abbr):
    """Return (opponent, 'vs'/'@', date) for the team's next unplayed game."""
    r = requests.get(SCHEDULE_URL.format(team_abbr_lower=player_team), headers=HEADERS)
    r.raise_for_status()
    for ev in r.json()["events"]:
        comp = ev["competitions"][0]
        if comp["status"]["type"]["completed"]:
            continue
        opp, at_vs = None, None
        for c in comp["competitors"]:
            if c["team"]["abbreviation"].upper() == team_abbr.upper():
                at_vs = "vs" if c["homeAway"] == "home" else "@"
            else:
                opp = c["team"]["abbreviation"]
        return opp, at_vs, ev["date"]
    return None, None, None  # no games left

next_opp, next_at_vs, next_date = get_next_opponent(player_team)

current_season = max(ranks)  # latest season already in `ranks`
next_def_rank = ranks[current_season].get(next_opp)

print(f"Next: {next_at_vs} {next_opp} on {next_date} | defense rank: {next_def_rank}")

Next: @ LV on 2026-10-18T20:25Z | defense rank: 23


In [ ]:
print(df_reg.shape)
print(f"{df_reg.memory_usage(deep=True).sum() / 1024:.1f} KB")

(38, 24)
44.3 KB


In [ ]:
df_reg.columns

Index(['rushingAttempts', 'rushingYards', 'yardsPerRushAttempt',
       'rushingTouchdowns', 'longRushing', 'receptions', 'receivingTargets',
       'receivingYards', 'yardsPerReception', 'receivingTouchdowns',
       'longReception', 'fumbles', 'fumblesLost', 'fumblesForced',
       'kicksBlocked', 'season', 'season_type', 'eventId', 'date', 'opponent',
       'atVs', 'result', 'score', 'opp_def_rank'],
      dtype='object')